In [2]:
from stark_qa import load_skb
import regex as re
import ast
from framework import Framework
from triplet import TripletEnd, Triplet
from logger import Step1Result, Step2bResult, Step3Result, Step4bResult, Step4aResult
import torch
import vss

In [3]:
data_split = 'test'
llm_model = 'openai/gpt-oss-120b'
emb_model = 'text-embedding-ada-002'
configs_path = 'configs.json'

In [4]:
from stark_qa import load_qa
from llm_bridge import LlmBridge
from logger import Logger

dataset_name = 'prime'
kb = load_skb(dataset_name, download_processed=True)

qa_dataset = load_qa('prime')

model_name = "openai/gpt-oss-120b"  # or "deepseek-chat", "gemini-pro", etc.
    
# Optional: Initialize logger if you want logging
logger = Logger("temp/")  # Adjust based on your Logger implementation

# # Create LLM Bridge instance
# llm = LlmBridge(model_name=model_name, logger=logger)

# framework = Framework("cypher_notebook", dataset_name, data_split, llm_model=llm_model, enable_vss=True,
#                   emb_model=emb_model, configs_path=configs_path)
# alpha = framework.settings.get("alpha")

# nodes_to_consider = str(framework.skb_b.skb.node_type_lst()).replace("'","")
# edges_to_consider = str(list(framework.settings.configs.get("edge_type_long2short").keys())).replace("'","")
# properties_to_consider = str(list(framework.settings.configs.get("avail_node_properties").keys())).replace("'","")

Loading from /home/sarthak/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/skb/prime/processed!
Use file from /home/sarthak/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/qa/prime/stark_qa/stark_qa_human_generated_eval.csv.


In [5]:
qa_dataset = load_qa('prime')
qa = qa_dataset.split_indices[data_split].reshape(-1).tolist()
qa = qa[:int(len(qa) * 0.1)]
test_queries = [qa_dataset[i] for i in qa]

Use file from /home/sarthak/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/qa/prime/stark_qa/stark_qa_human_generated_eval.csv.


In [6]:
print(test_queries[16])

('Find medical conditions linked to the gene SLC29A3 that should not be treated with the drug Goserelin.', 11077, [29113], None)


In [9]:
print(kb.get_doc_info(29113,add_rel=True))

- name: diabetic ketoacidosis
- type: disease
- source: MONDO
- details:
  - mondo_name: diabetic ketoacidosis
  - mondo_definition: The metabolic condition resulted from uncontrolled diabetes mellitus, in which the shift of acid-base status of the body toward the acid side because of loss of base or retention of acids other than carbonic acid is accompanied by the accumulation of ketone bodies in body tissues and fluids.
  - umls_description: A heterogeneous group of disorders characterized by hyperglycemia and glucose intolerance.
  - mayo_symptoms: Diabetic ketoacidosis signs and symptoms often develop quickly, sometimes within 24 hours. For some, these signs and symptoms may be the first indication of having diabetes. You may notice: Excessive thirst, Frequent urination, Nausea and vomiting, Stomach pain, Weakness or fatigue, Shortness of breath, Fruity-scented breath, Confusion, More-specific signs of diabetic ketoacidosis — which can be detected through home blood and urine testi

In [10]:
print(kb.get_doc_info(16468,add_rel=True))

- name: Goserelin
- type: drug
- source: DrugBank
- details:
  - description: Goserelin is a synthetic hormone. In men, it stops the production of the hormone testosterone, which may stimulate the growth of cancer cells. In women, goserelin decreases the production of the hormone estradiol (which may stimulate the growth of cancer cells) to levels similar to a postmenopausal state. When the medication is stopped, hormone levels return to normal.
  - half_life: The half-life is 4-5 hours
  - indication: Goserelin is indicated for:
  - mechanism_of_action: Goserelin is a synthetic decapeptide analogue of LHRH. Goserelin acts as a potent inhibitor of pituitary gonadotropin secretion when administered in the biodegradable formulation. The result is sustained suppression of LH and serum testosterone levels.
  - protein_binding: 27.3%
  - pharmacodynamics: The pharmacokinetics of goserelin have been determined in both male and female healthy volunteers and patients. In these studies, goserel

In [14]:
print(kb.get_rel_info(16468))

- relations:
  target: {gene/protein: (GNRHR, LHCGR),}
  contraindication: {disease: (monogenic obesity, diabetic ketoacidosis, diabetes mellitus (disease), hypertensive disorder, obesity disorder, hypercalcemia disease, phosphorus metabolism disease, hypertension, urinary tract obstruction, nephrocalcinosis, hyperglycemia),}
  indication: {disease: (endometriosis of uterus, endometriosis (disease)),}
  synergistic_interaction: {drug: (Fluocinolone acetonide, Prednisone, Budesonide, Liothyronine, Diclofenac, Diflunisal, Dimethyl sulfoxide, Hydroxocobalamin, Tocopherol, Chromium, Chromic citrate, Chromic nitrate, Chromium gluconate, Chromium nicotinate, Chromous sulfate, Octreotide, Icosapent, Pyridoxine, Torasemide, Nelfinavir, Butabarbital, Benzatropine, Ziprasidone, Metoprolol, Topiramate, Cefmetazole, Conjugated estrogens, Atomoxetine, Etonogestrel, Chlorthalidone, Valproic acid, Acetaminophen, Amitriptyline, Hydromorphone, Indomethacin, Methadone, Olanzapine, Diltiazem, Alprazolam,

In [18]:
for doc in [37044,35583,38682,33656,36035,29113] :
    print(kb.get_doc_info(doc,add_rel=True))
    print('-------------------\n\n')
    

- name: urinary tract obstruction
- type: disease
- source: MONDO
- details:
  - mondo_name: urinary tract obstruction
  - mondo_definition: Blockage of the normal flow of contents of the urinary tract.
  - umls_description: A disorder characterized by blockage of the normal flow of contents of the urinary tract.
- relations:
  contraindication: {drug: (Benzatropine, Morphine, Acyclovir, Pseudoephedrine, Ephedrine, Caffeine, Profenamine, Ambenonium, Neostigmine, Phenylephrine, Chlorpheniramine, Phenobarbital, Prochlorperazine, Dextromethorphan, Biperiden, Hydrocodone, Dexchlorpheniramine, Sulfadiazine, Ergotamine, Milnacipran, Homatropine, Sulfisoxazole, Sulfasalazine, Trihexyphenidyl, Procyclidine, Hyoscyamine, Homatropine methylbromide, Scopolamine, Brompheniramine, Bethanechol, Isopropamide, Mepenzolate, Methacholine, Potassium chloride, Clidinium, Propantheline, Dicyclomine, Flavoxate, Goserelin, Pheniramine, Leuprolide, Triptorelin, Pyridostigmine, Physostigmine, Rivastigmine, Edr

In [30]:
print(kb.get_neighbor_nodes(21961,edge_type="associated with"))


[22304, 24199, 27219, 27933, 28407, 28552, 29113, 30438, 30877, 31109, 33528, 33616, 33636, 36043, 36453, 36923, 37703, 38898, 83760, 83840]
